<a href="https://colab.research.google.com/github/Da-rell/ProjectAI_ExpertSystem_A/blob/main/RAG_Privacy_Policy_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Cell 1 — Install Dependencies

In [ ]:
# Install semua library yang dibutuhkan
!pip install -q pymupdf sentence-transformers chromadb gradio google-generativeai

print("✅ Semua library berhasil diinstall!")

##Cell 2 — Konfigurasi API Key & Inisialisasi Model

In [ ]:
import fitz  # PyMuPDF
import chromadb
import gradio as gr
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from google.colab import files, userdata
import re
import os

# KONFIGURASI API KEY
# Pilih salah satu cara di bawah ini:

# CARA 1 (Rekomendasi): Simpan di Colab Secrets
# Klik ikon 🔑 di sidebar kiri Colab → Add new secret
# Name: GEMINI_API_KEY | Value: (api key kamu)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ API Key berhasil dibaca dari Colab Secrets!")
except:
    # CARA 2 (Alternatif): Isi langsung di sini
    GEMINI_API_KEY = "MASUKKAN_API_KEY_GEMINI_KAMU_DISINI"
    print("⚠️  Menggunakan API Key yang di-hardcode. Pertimbangkan pakai Colab Secrets.")


# Inisialisasi Gemini
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("models/gemini-2.5-flash")
print("✅ Gemini API siap!")

# Inisialisasi Embedding Model (berjalan lokal di Colab, gratis)
print("⏳ Loading embedding model (mungkin butuh 1-2 menit pertama kali)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model siap!")

# Inisialisasi ChromaDB
chroma_client = chromadb.Client()
COLLECTION_NAME = "rag_privacy_policy"
print("✅ ChromaDB siap!")

##Cell 3 — Upload & Proses PDF

In [ ]:
# Fungsi-fungsi pemrosesan dokumen

def extract_text_from_pdf(pdf_path: str) -> str:
    """Ekstrak semua teks dari file PDF menggunakan PyMuPDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text()
        full_text += f"\n--- Halaman {page_num + 1} ---\n{text}"
    doc.close()
    return full_text


def chunk_text(text: str, chunk_size: int = 400, overlap: int = 80) -> list:
    """
    Potong teks menjadi chunk-chunk kecil dengan overlap.
    - chunk_size: jumlah kata per chunk
    - overlap: jumlah kata yang tumpang tindih antar chunk
      (menjaga konteks agar tidak hilang di batas chunk)
    """
    text = re.sub(r'\n+', ' ', text).strip()
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if len(chunk.strip()) > 50:  # abaikan chunk yang terlalu pendek
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


def build_vector_store(chunks: list, collection_name: str) -> object:
    """Buat embedding untuk setiap chunk dan simpan ke ChromaDB."""
    # Hapus collection lama jika ada (untuk re-run)
    try:
        chroma_client.delete_collection(name=collection_name)
    except:
        pass

    collection = chroma_client.create_collection(name=collection_name)

    print(f"⏳ Membuat embeddings untuk {len(chunks)} chunks...")
    embeddings = embedder.encode(chunks, show_progress_bar=True)

    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

    print(f"✅ {len(chunks)} chunks berhasil disimpan ke ChromaDB!")
    return collection


# Upload PDF dari komputer lokal
print("📂 Silakan upload file PDF kamu:")
uploaded = files.upload()

# Ambil nama file yang di-upload
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ File '{pdf_filename}' berhasil di-upload!")

# Proses PDF
print("\n📄 Mengekstrak teks dari PDF...")
raw_text = extract_text_from_pdf(pdf_filename)
print(f"✅ Total karakter: {len(raw_text):,}")

print("\n✂️  Memotong teks menjadi chunks...")
chunks = chunk_text(raw_text)
print(f"✅ Total chunks: {len(chunks)}")

# Simpan ke vector store
print("\n🗄️  Menyimpan ke ChromaDB...")
collection = build_vector_store(chunks, COLLECTION_NAME)

print(f"\n🎉 Dokumen '{pdf_filename}' siap digunakan!")

##Cell 4 — Core RAG Pipeline

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list:
    """
    RETRIEVAL: Cari chunk paling relevan menggunakan semantic search.
    """
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0]


def generate(query: str, context_chunks: list) -> str:
    """
    GENERATION: Kirim context + query ke Gemini, dapatkan jawaban.
    """
    context = "\n\n".join(context_chunks)

    prompt = f"""Kamu adalah analis dokumen hukum yang membantu pengguna memahami \
Kebijakan Privasi dan Syarat & Ketentuan (Terms of Service) dalam bahasa yang mudah dipahami.

Berikut adalah potongan dokumen yang relevan sebagai konteks:
============================================================
{context}
============================================================

Pertanyaan pengguna: {query}

Instruksi:
- Jawab HANYA berdasarkan dokumen di atas
- Gunakan bahasa Indonesia yang mudah dipahami orang awam
- Jika informasi tidak ada di dokumen, katakan dengan jujur
- Sertakan referensi bagian dokumen yang relevan
- Gunakan format poin-poin jika jawabannya kompleks

Jawaban:"""

    response = gemini_model.generate_content(prompt)
    return response.text


def rag(query: str) -> tuple:
    """
    Pipeline RAG lengkap: Query → Retrieve → Generate → Answer
    Returns: (answer, context_preview)
    """
    if not query.strip():
        return "⚠️ Pertanyaan tidak boleh kosong.", ""

    relevant_chunks = retrieve(query)
    answer = generate(query, relevant_chunks)

    context_preview = "\n\n" + "─" * 50 + "\n\n".join(
        [f"📌 Chunk {i+1}:\n{chunk[:300]}..." for i, chunk in enumerate(relevant_chunks)]
    )

    return answer, context_preview


# Test pipeline sebelum launch UI
print("🧪 Testing RAG pipeline...")
test_q = "Berapa usia minimum untuk menggunakan YouTube?"
test_ans, _ = rag(test_q)
print(f"\nQ: {test_q}")
print(f"A: {test_ans}")
print("\n✅ Pipeline RAG berjalan dengan baik!")

##Cell 5 — Launch Gradio UI

In [ ]:
# Contoh pertanyaan untuk inspirasi pengguna
EXAMPLES = [
    ["Apakah YouTube bisa menghapus konten saya tanpa pemberitahuan?"],
    ["Berapa usia minimum untuk menggunakan YouTube?"],
    ["Apa yang terjadi jika akun saya ditangguhkan?"],
    ["Apakah YouTube bisa memonetisasi video saya tanpa izin saya?"],
    ["Apa hak saya sebagai kreator konten di YouTube?"],
    ["Bagaimana cara menghapus akun YouTube saya?"],
    ["Apa yang dimaksud dengan teguran (strike) di YouTube?"],
    ["Apakah saya tetap memiliki hak cipta atas video yang saya upload?"],
]


# Event handler functions
def chat(message: str, history: list) -> tuple:
    """Handler untuk chatbot dengan history percakapan."""
    if not message.strip():
        return "", history
    answer, _ = rag(message)
    history.append((message, answer))
    return "", history


def get_context(query: str) -> str:
    """Handler untuk menampilkan context yang di-retrieve."""
    if not query.strip():
        return "Masukkan pertanyaan terlebih dahulu."
    _, ctx = rag(query)
    return ctx


def fill_example(example_query: str) -> str:
    """Isi input dengan contoh pertanyaan."""
    return example_query


# Build Gradio Interface
CSS = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.tab-nav button { font-size: 15px; font-weight: 600; }
.example-btn { font-size: 13px !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"), css=CSS,
               title="RAG - Privacy Policy Analyst") as demo:

    # Header
    gr.Markdown("""
    # 🔍 Privacy Policy & Terms of Service Analyst
    <p style='font-size:16px; color:#555;'>
    Powered by <b>RAG</b> (Retrieval-Augmented Generation) · Gemini 2.0 Flash · ChromaDB
    </p>
    Tanyakan apa saja tentang dokumen yang telah di-upload menggunakan bahasa sehari-hari.
    Sistem akan <b>mencari bagian yang relevan</b> lalu menjawab dalam bahasa yang mudah dipahami.
    """)

    gr.Markdown(f"📄 **Dokumen aktif:** `{pdf_filename}`")

    # Tabs
    with gr.Tabs():

        # TAB 1: Chatbot
        with gr.TabItem("💬 Chatbot"):
            chatbot_ui = gr.Chatbot(
                label="Percakapan",
                height=420,
                bubble_full_width=False,
                avatar_images=(None, "https://i.imgur.com/7k12EPD.png")
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Contoh: Apakah YouTube bisa menjual data saya ke pihak ketiga?",
                    label="Pertanyaan Anda",
                    lines=2,
                    scale=5
                )
                with gr.Column(scale=1):
                    send_btn = gr.Button("Kirim 🚀", variant="primary", size="lg")
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="sm")

            gr.Markdown("### 💡 Coba Pertanyaan Ini:")
            gr.Examples(
                examples=EXAMPLES,
                inputs=msg_box,
                label="",
                examples_per_page=4
            )

            # Events
            send_btn.click(chat, [msg_box, chatbot_ui], [msg_box, chatbot_ui])
            msg_box.submit(chat, [msg_box, chatbot_ui], [msg_box, chatbot_ui])
            clear_btn.click(lambda: ([], ""), outputs=[chatbot_ui, msg_box])

        # TAB 2: Context Viewer
        with gr.TabItem("🔎 Lihat Context RAG"):
            gr.Markdown("""
            Tab ini menampilkan **potongan dokumen asli** yang diambil sistem RAG
            sebagai dasar pembuatan jawaban. Berguna untuk **verifikasi akurasi** dan presentasi.
            """)
            ctx_input = gr.Textbox(
                placeholder="Masukkan pertanyaan untuk melihat dokumen yang di-retrieve...",
                label="Pertanyaan"
            )
            ctx_btn = gr.Button("🔍 Retrieve Context", variant="primary")
            ctx_output = gr.Textbox(
                label="📚 Chunks dari ChromaDB (Top-5 paling relevan)",
                lines=22,
                interactive=False
            )
            ctx_btn.click(get_context, ctx_input, ctx_output)
            ctx_input.submit(get_context, ctx_input, ctx_output)

        # TAB 3: Arsitektur
        with gr.TabItem("📊 Arsitektur Sistem"):
            gr.Markdown("""
            ## Cara Kerja RAG

            ### Phase 1 — Indexing (dilakukan sekali saat startup)
            ```
            PDF File
               ↓  PyMuPDF
            Raw Text
               ↓  Chunking (400 kata, overlap 80)
            Text Chunks []
               ↓  Sentence Transformers (all-MiniLM-L6-v2)
            Embeddings (384-dim vectors)
               ↓
            ChromaDB (Vector Store)
            ```

            ### Phase 2 — Query (setiap kali user bertanya)
            ```
            User Query
               ↓  Sentence Transformers
            Query Embedding
               ↓  Cosine Similarity Search
            Top-5 Relevant Chunks
               ↓  Prompt Engineering
            Context + Query → Gemini 1.5 Flash
               ↓
            Jawaban dalam Bahasa Indonesia
            ```

            ## Tech Stack
            | Komponen | Library | Keterangan |
            |---|---|---|
            | PDF Parser | PyMuPDF (fitz) | Ekstrak teks dari PDF |
            | Embedding | sentence-transformers | Konversi teks → vector |
            | Vector DB | ChromaDB | Simpan & cari embedding |
            | LLM | Gemini 2.0 Flash | Generate jawaban |
            | UI | Gradio | Interface chatbot |
            | Platform | Google Colab | Runtime gratis |

            ## Mengapa RAG?
            - ✅ LLM tidak perlu tahu seluruh dokumen — cukup bagian relevan
            - ✅ Jawaban selalu bersumber dari dokumen asli (bukan halusinasi)
            - ✅ Bisa diupdate dokumennya tanpa re-training model
            - ✅ Hemat token karena hanya mengirim chunk relevan ke LLM
            """)

# 🚀 Launch!
# share=True → generate public URL yang bisa diakses siapa saja
# debug=True  → tampilkan error di UI jika ada
print("\n🚀 Meluncurkan Gradio...")
print("⏳ Tunggu beberapa detik hingga link muncul di bawah ini...\n")

demo.launch(share=True, debug=True)